# Auto Tagging Support Tickets Using Google Gemini

## Objective

Automatically classify customer support tickets into relevant categories using Google's Gemini Large Language Model.

## Dataset

Support Ticket Dataset

## Workflow

1. Load Dataset
2. Preprocess Text
3. Zero-shot Classification
4. Few-shot Classification
5. Compare Results
6. Output Top 3 Tags

## Technologies

- Google Gemini API
- Python
- Pandas
- Google Colab



In [8]:
!pip install -q -U google-genai

In [9]:
from google import genai

In [39]:
from google.colab import userdata
API_KEY = userdata.get('secretName')
client = genai.Client(api_key=API_KEY)

### Load Dataset

In [11]:
import pandas as pd
df = pd.read_csv("customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


### Basic Information

In [38]:
print(df.shape)
df.info()

(8469, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Ticket Subject      8469 non-null   object
 1   Ticket Description  8469 non-null   object
 2   Ticket Type         8469 non-null   object
 3   text                8469 non-null   object
dtypes: object(4)
memory usage: 264.8+ KB


### Missing Values

In [13]:
df.isnull().sum()

,0
Ticket ID,0
Customer Name,0
Customer Email,0
Customer Age,0
Customer Gender,0
Product Purchased,0
Date of Purchase,0
Ticket Type,0
Ticket Subject,0
Ticket Description,0


### Keep Only Required Columns

In [15]:
df = df[[
    "Ticket Subject",
    "Ticket Description",
    "Ticket Type"
]]
df.head()

,Ticket Subject,Ticket Description,Ticket Type
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry


### Check Ticket Types

In [16]:
df["Ticket Type"].value_counts()

,count
Ticket Type,
Refund request,1752
Technical issue,1747
Cancellation request,1695
Product inquiry,1641
Billing inquiry,1634


### Create Input Text

In [17]:
df["text"] = (
    df["Ticket Subject"] +
    ". " +
    df["Ticket Description"]
)
df[["text","Ticket Type"]].head()

,text,Ticket Type
0,Product setup. I'm having an issue with the {p...,Technical issue
1,Peripheral compatibility. I'm having an issue ...,Technical issue
2,Network problem. I'm facing a problem with my ...,Technical issue
3,Account access. I'm having an issue with the {...,Billing inquiry
4,Data loss. I'm having an issue with the {produ...,Billing inquiry


### Test Gemini Connection

In [40]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello!"
)
print(response.text)

Hello!


### Zero-Shot Classification Function

In [19]:
import json
import re

def zero_shot_classifier(ticket_text):
    prompt = f"""
You are an expert AI support ticket classifier.
Classify the support ticket into the THREE most relevant categories.
Available Categories:
{', '.join(TAGS)}

IMPORTANT:
- Return ONLY valid JSON.
- Use ONLY categories from the list above.

JSON format:
{{
  "tags": [
    "Category 1",
    "Category 2",
    "Category 3"
  ]
}}

Support Ticket:
{ticket_text}
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    text = response.text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        data = json.loads(text)
        return data["tags"]
    except Exception as e:
        print("Error:", e)
        print(text)
        return []

### Test Zero-Shot

In [41]:
TAGS = df["Ticket Type"].unique().tolist()
print(TAGS)

['Technical issue', 'Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request']


### Run Zero-Shot Classification

In [42]:
sample_df = df.head(5).copy()
sample_df["Zero Shot Tags"] = sample_df["text"].apply(zero_shot_classifier)
sample_df.head()

,Ticket Subject,Ticket Description,Ticket Type,text,Zero Shot Tags
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue,Product setup. I'm having an issue with the {p...,"[Technical issue, Billing inquiry, Product inq..."
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,Peripheral compatibility. I'm having an issue ...,"[Technical issue, Product inquiry, Refund requ..."
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Network problem. I'm facing a problem with my ...,"[Technical issue, Product inquiry, Refund requ..."
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Account access. I'm having an issue with the {...,"[Technical issue, Product inquiry, Billing inq..."
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Data loss. I'm having an issue with the {produ...,"[Technical issue, Product inquiry, Refund requ..."


### Few-Shot Classifier

In [43]:
def few_shot_classifier(ticket_text):
    prompt = f"""
You are an expert customer support ticket classifier.
Available Categories:

{', '.join(TAGS)}

Examples
Ticket:
"I forgot my password and cannot access my account."

Output
{{
"tags":[
"Technical Support",
"Account Access",
"Password Reset"
]
}}
----------------------
Ticket:
"I was charged twice for my premium subscription."

Output
{{
"tags":[
"Billing",
"Refund",
"Payment Issue"
]
}}

----------------------

Now classify this ticket.
Return ONLY JSON.
Ticket:

{ticket_text}
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    text = response.text.strip()
    text = text.replace("```json","").replace("```","").strip()
    try:
        return json.loads(text)["tags"]
    except:
        return []

### Run Few-Shot

In [44]:
sample_df["Few Shot Tags"] = sample_df["text"].apply(few_shot_classifier)
sample_df.head()

,Ticket Subject,Ticket Description,Ticket Type,text,Zero Shot Tags,Few Shot Tags
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue,Product setup. I'm having an issue with the {p...,"[Technical issue, Billing inquiry, Product inq...","[Technical issue, Product Setup, Troubleshooti..."
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,Peripheral compatibility. I'm having an issue ...,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Peripheral Compatibility, Pr..."
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Network problem. I'm facing a problem with my ...,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Device Malfunction, Power Is..."
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Account access. I'm having an issue with the {...,"[Technical issue, Product inquiry, Billing inq...","[Technical Support, Account Access, Product Is..."
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Data loss. I'm having an issue with the {produ...,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Data Loss, Battery Issue, Pe..."


### Compare Results

In [45]:
comparison = sample_df[[
    "Ticket Type",
    "Zero Shot Tags",
    "Few Shot Tags"
]]
comparison.head(5)

,Ticket Type,Zero Shot Tags,Few Shot Tags
0,Technical issue,"[Technical issue, Billing inquiry, Product inq...","[Technical issue, Product Setup, Troubleshooti..."
1,Technical issue,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Peripheral Compatibility, Pr..."
2,Technical issue,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Device Malfunction, Power Is..."
3,Billing inquiry,"[Technical issue, Product inquiry, Billing inq...","[Technical Support, Account Access, Product Is..."
4,Billing inquiry,"[Technical issue, Product inquiry, Refund requ...","[Technical issue, Data Loss, Battery Issue, Pe..."


### Compare Predictions with Actual Labels

In [47]:
sample_df["Top Prediction"] = sample_df["Zero Shot Tags"].apply(
    lambda x: x[0] if len(x) > 0 else None
)
comparison = sample_df[[
    "Ticket Type",
    "Top Prediction",
    "Zero Shot Tags"
]]
comparison

,Ticket Type,Top Prediction,Zero Shot Tags
0,Technical issue,Technical issue,"[Technical issue, Billing inquiry, Product inq..."
1,Technical issue,Technical issue,"[Technical issue, Product inquiry, Refund requ..."
2,Technical issue,Technical issue,"[Technical issue, Product inquiry, Refund requ..."
3,Billing inquiry,Technical issue,"[Technical issue, Product inquiry, Billing inq..."
4,Billing inquiry,Technical issue,"[Technical issue, Product inquiry, Refund requ..."


### Calculate Accuracy

In [48]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(
    sample_df["Ticket Type"],
    sample_df["Top Prediction"]
)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 60.00%


### Save Results

In [50]:
comparison.to_csv("ticket_predictions.csv", index=False)
print("Predictions saved successfully!")

Predictions saved successfully!


# Conclusion

## Summary

This project uses prompt engineering with the Gemini LLM to perform automatic ticket tagging. Zero-shot and few-shot prompting strategies were evaluated, with few-shot prompting improving the consistency of predictions by providing task-specific examples. Full model fine-tuning was not performed because the selected LLM workflow focused on prompt-based adaptation.

### Key Steps

- Loaded and explored a real-world support ticket dataset.
- Created combined ticket text from the subject and description.
- Used zero-shot prompting to classify support tickets.
- Used few-shot prompting with examples to improve predictions.
- Generated the top three predicted tags for each ticket.
- Compared predicted tags with the original ticket categories.

### Outcome

The project shows how prompt engineering can automate support ticket routing without training a custom machine learning model. Few-shot prompting provides additional context to the LLM and can improve the consistency of predictions compared to zero-shot prompting.

This workflow can be integrated into customer support systems to assist agents in automatically categorizing incoming tickets.